# Module 14 · Trajectory

Slingshot pseudotime on the transcriptional manifold, with scores overlaid
afterwards.

**The circularity constraint, and why it shapes every section here.** If the
trajectory is fitted on a senescence-informed embedding, or rooted at the cell
that scores highest for senescence, then "senescence increases along
pseudotime" is a restatement of the fit, not a finding. So:

- the trajectory is fitted on **clusters and the transcriptional manifold
  only** — no score enters the fit;
- the root is chosen at the **homeostatic** end, defined by the homeostatic
  panel, which is not the quantity being tested;
- scores are overlaid **post hoc**, for visualization and correlation, never as
  input.

**Raw scores, not composite axes.** A composite of the form
`Score_state − Score_Homeostatic` shares its `−Homeostatic` term across states,
which makes different state axes correlate with each other and with anything
else that carries the same term. That is acceptable for quadrant definitions in
module 10; it is not acceptable for a correlation against pseudotime, which is
what this module computes. Raw scores throughout.

| Section | |
|---|---|
| 01-02 | config, coarse clusters and root selection |
| 03-04 | Slingshot fit, lineage validity filter |
| 05-07 | branching tree, per-lineage UMAPs, score ~ pseudotime |
| 08-09 | cell density along pseudotime, progression panels |

---
## 01 · Config

**Why.** Self-contained — the notebook carries its own config rather than
importing one, so it can be read and run without tracing an import elsewhere.
Paths come from the environment; see `.env.example`.

**`AXIS` is the one thing you change.** Column names, quadrant labels, figure
titles, contrast names and output directories all derive from it. Outputs are
namespaced by axis so two runs never overwrite each other.

```
AXIS <- "IRM"    # "IRM" | "DAM_like" | "ARM" | "Stress"
```

`STATE_ORDER` and `STATE_COLORS` list all five states regardless of `AXIS` —
those are the annotation registry, not a per-run choice.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
# Paths come from the environment - see .env.example. Nothing below hardcodes
# a filesystem location.
#   SENESCENCE_DATA : analysis root (module outputs written under it)
#   SENESCENCE_REF  : reference root (published panels, read-only)
# =============================================================================

SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")
if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")


# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = file.path(REF_DIR, "markers"),
    sloan_xlsx     = file.path(REF_DIR, "markers", "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(REF_DIR, "markers", "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(REF_DIR, "markers", "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)



# =============================================================================
# §0.2 — AXIS SELECT
# =============================================================================
# The ONE thing you change to re-run the whole flow on a different state.
# Everything downstream — column names, quadrant labels, figure titles,
# legends, output paths — derives from this. Nothing is hardcoded per state.
# =============================================================================

AXIS <- "IRM"          # <<< "IRM" | "DAM_like" | "ARM" | "Stress"

# --- registry: score column candidates + display label + canonical hex -------
# Score columns are looked up in order; the first present on the object wins.
AXIS_REGISTRY <- list(
    # tag = short token used in CONTRAST NAMES and therefore in GSEA/DE filenames.
    # It is deliberately NOT the same as `lab` (display) or the list key: existing
    # results on disk are named DAMaxis_*, not DAM_likeaxis_*.
    IRM      = list(cols = c("Score_IRM",      "IRM1"),      lab = "IRM",      tag = "IRM",    hex = "#2980B9"),
    DAM_like = list(cols = c("Score_DAM_like", "DAM_like1"), lab = "DAM-like", tag = "DAM",    hex = "#C0392B"),
    ARM      = list(cols = c("Score_ARM",      "ARM1"),      lab = "ARM",      tag = "ARM",    hex = "#E67E22"),
    Stress   = list(cols = c("Score_Stress",   "Stress1"),   lab = "Stress",   tag = "Stress", hex = "#8E44AD")
)
stopifnot(AXIS %in% names(AXIS_REGISTRY))

AXIS_SPEC <- AXIS_REGISTRY[[AXIS]]
X_LAB     <- AXIS_SPEC$lab
X_HEX     <- AXIS_SPEC$hex
Y_LAB     <- "Senescence"
Y_TAG     <- "SnC"
X_TAG     <- AXIS_SPEC$tag
SEN_CANDIDATES <- c("senescence_score", "SenePy_score")

# --- quadrant labels derive from the axis -----------------------------------
QUAD_COL    <- paste0("quad4_", tolower(AXIS))
QUAD_LEVELS <- c(sprintf("Sen- %s-", X_LAB), sprintf("Sen+ %s-", X_LAB),
                 sprintf("Sen- %s+", X_LAB), sprintf("Sen+ %s+", X_LAB))
QUAD_COLORS <- setNames(c("#B8B8B8", "#2E7D32", X_HEX, "#6A1B9A"), QUAD_LEVELS)

# --- the four DE / GSEA contrasts, named off the tags ------------------------
CONTRASTS <- c(sprintf("%saxis_%spos", X_TAG, Y_TAG),   # axis effect within SnC+
               sprintf("%saxis_%sneg", X_TAG, Y_TAG),   # axis effect within SnC-
               sprintf("%saxis_%spos", Y_TAG, X_TAG),   # sen effect within axis+
               sprintf("%saxis_%sneg", Y_TAG, X_TAG))   # sen effect within axis-
CONTRAST_HEADERS <- c(
    sprintf("%s+%s+ vs %s+%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s−%s+ vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s+ vs %s−%s+", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s− vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB))

# --- outputs are namespaced by axis so runs never overwrite each other -------
AXIS_FIG_DIR <- file.path(PATHS$figures, AXIS)
AXIS_RES_DIR <- file.path(PATHS$results, AXIS)
dir.create(AXIS_FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(AXIS_RES_DIR, recursive = TRUE, showWarnings = FALSE)

# --- state annotation (all five; NOT gated by AXIS) --------------------------
STATE_ORDER  <- c("Homeostatic", "ARM", "IRM", "Stress", "DAM_like")
STATE_COLORS <- c(Homeostatic = "#7F8C8D", ARM = "#E67E22", IRM = "#2980B9",
                  Stress = "#8E44AD", DAM_like = "#C0392B")

# --- DE model design (confirmed 2026-07-28; supersedes the leaner ~pop+grp2+Sex)
DE_DESIGN     <- "~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort"
DE_COEF       <- "popTEST"
DE_MIN_CELLS  <- 10L

# --- GSEA (consumed by the python notebook via gsea_config.json) -------------
GSEA_DBS      <- c("Reactome_2022")
GSEA_FDR_SIG  <- 0.05
GSEA_N_COMMON <- 6L      # sig in >=3 contrasts, top N by mean NES
GSEA_N_UNIQUE <- 4L      # sig in exactly 1 contrast, top N by |NES|
GSEA_RIBO_STRIP <- FALSE # keep translational terms; see Part F Why



# §0.3 — AXIS-AWARE HELPERS  (one definition each — see note)
# =============================================================================
# In the source notebook fit_one was defined 13x, resolve_score_col 6x,
# z_score 5x, venn2 4x, and save_figure was REDEFINED at cells 317/341,
# shadowing the canonical version above. Everything lives here now so a
# later cell cannot silently shadow it.
# =============================================================================

# --- resolve a score column from candidates ---------------------------------
resolve_score_col <- function(md, candidates, what = "score") {
    hit <- candidates[candidates %in% colnames(md)]
    if (!length(hit)) stop(sprintf("no %s column found; tried: %s",
                                   what, paste(candidates, collapse = ", ")))
    hit[1]
}

z_score <- function(x) as.numeric(scale(x))

# --- build the SnC x AXIS quadrant column -----------------------------------
# Mean-split on z-scored values, exactly as the source (cell 54).
build_quadrants <- function(obj, axis = AXIS, verbose = TRUE) {
    md   <- obj@meta.data
    spec <- AXIS_REGISTRY[[axis]]
    sen  <- resolve_score_col(md, SEN_CANDIDATES, "senescence")
    xcol <- resolve_score_col(md, spec$cols, paste(axis, "score"))
    sz <- z_score(md[[sen]]); xz <- z_score(md[[xcol]])
    lab <- spec$lab
    q <- ifelse(sz >  0 & xz >  0, sprintf("Sen+ %s+", lab),
        ifelse(sz >  0 & xz <= 0, sprintf("Sen+ %s-", lab),
        ifelse(sz <= 0 & xz >  0, sprintf("Sen- %s+", lab),
                                  sprintf("Sen- %s-", lab))))
    q[is.na(sz) | is.na(xz)] <- NA
    obj[[paste0("quad4_", tolower(axis))]] <- q
    obj$sen_z <- sz
    obj$axis_z <- xz
    if (verbose) {
        cat(sprintf("  scores : sen=%s  %s=%s\n", sen, axis, xcol))
        print(table(q, useNA = "ifany"))
        cat(sprintf("  cor(sen_z, %s_z) = %.3f   <- independence check\n",
                    tolower(axis), cor(sz, xz, use = "complete.obs")))
    }
    obj
}

# --- PREFLIGHT: fail loudly before any model runs ---------------------------
preflight_axis <- function(obj, axis = AXIS, min_cells = 50L, donor_col = "Donor") {
    md <- obj@meta.data; ok <- TRUE
    say <- function(pass, msg) {
        cat(sprintf("  [%s] %s\n", if (pass) "OK  " else "FAIL", msg))
        if (!pass) ok <<- FALSE
    }
    cat(sprintf("\n── PREFLIGHT · axis = %s ──\n", axis))
    say(axis %in% names(AXIS_REGISTRY), sprintf("axis '%s' is registered", axis))
    spec <- AXIS_REGISTRY[[axis]]
    xhit <- spec$cols[spec$cols %in% colnames(md)]
    say(length(xhit) > 0, sprintf("score column present (%s)",
        if (length(xhit)) xhit[1] else paste(spec$cols, collapse = "/")))
    shit <- SEN_CANDIDATES[SEN_CANDIDATES %in% colnames(md)]
    say(length(shit) > 0, "senescence score column present")
    if (length(xhit) && length(shit)) {
        say(sd(md[[xhit[1]]], na.rm = TRUE) > 0, "axis score is non-constant")
        say(sd(md[[shit[1]]], na.rm = TRUE) > 0, "senescence score is non-constant")
    }
    qc <- paste0("quad4_", tolower(axis))
    if (qc %in% colnames(md)) {
        tb <- table(md[[qc]])
        say(length(tb) == 4, sprintf("all four quadrants populated (%d)", length(tb)))
        say(all(tb >= min_cells), sprintf("every quadrant >= %d cells (min %d)",
                                          min_cells, min(tb)))
        if (donor_col %in% colnames(md)) {
            nd <- tapply(md[[donor_col]], md[[qc]], function(z) length(unique(z)))
            say(all(nd >= 2), sprintf("every quadrant has >=2 donors (min %d)", min(nd)))
        }
    } else cat(sprintf("  [--  ] %s not built yet (run B1)\n", qc))
    cat(sprintf("── %s ──\n\n", if (ok) "PASS" else "STOP: fix before proceeding"))
    invisible(ok)
}

# --- ONE mixed-model fitter (replaces 13 copies of fit_one) -----------------
# formula_str is built by the caller, so every Part C analysis is this
# function with a different formula and a different subset.
fit_lmm <- function(df, formula_str, term, label = NA_character_) {
    fit <- tryCatch(lmerTest::lmer(as.formula(formula_str), data = df,
                                   REML = TRUE,
                                   control = lme4::lmerControl(
                                       optimizer = "bobyqa",
                                       optCtrl = list(maxfun = 2e5))),
                    error = function(e) NULL, warning = function(w) NULL)
    if (is.null(fit)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    co <- summary(fit)$coefficients
    if (!term %in% rownames(co)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    b <- co[term, "Estimate"]; s <- co[term, "Std. Error"]
    data.frame(label = label, term = term, beta = b, se = s,
               ci_low = b - 1.96 * s, ci_high = b + 1.96 * s,
               p_value = co[term, "Pr(>|t|)"], n = nrow(df), converged = TRUE)
}

# --- ONE forest renderer (replaces 7 near-copies) ---------------------------
forest_plot <- function(res, title = "", xlab = "beta (95% CI)",
                        facet = NULL, color = X_HEX) {
    stopifnot(all(c("label", "beta", "ci_low", "ci_high") %in% names(res)))
    if (!"p_adj" %in% names(res))
        res$p_adj <- p.adjust(res$p_value, method = STATISTICAL_PARAMS$fdr_method)
    res$sig <- sig_stars(res$p_adj)
    res$label <- factor(res$label, levels = rev(unique(res$label)))
    p <- ggplot(res, aes(x = beta, y = label)) +
        geom_vline(xintercept = 0, linetype = "dashed",
                   colour = "grey60", linewidth = 0.3) +
        geom_errorbarh(aes(xmin = ci_low, xmax = ci_high),
                       height = 0, linewidth = 0.4, colour = color) +
        geom_point(size = 1.8, colour = color) +
        geom_text(aes(x = ci_high, label = sig), hjust = -0.35,
                  size = 2.6, na.rm = TRUE) +
        labs(title = title, x = xlab, y = NULL) +
        theme_clean() +
        theme(panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.25))
    if (!is.null(facet)) p <- p + facet_wrap(as.formula(paste("~", facet)), scales = "free_x")
    p + coord_cartesian(clip = "off")
}

# --- block banner: prints the Why with the axis resolved --------------------
say_block <- function(id, title, why = NULL) {
    cat("\n", strrep("═", 76), "\n", sep = "")
    cat(sprintf("%s  ·  %s\n", id, sprintf(title, X_LAB)))
    cat(strrep("═", 76), "\n", sep = "")
    if (!is.null(why)) cat(sprintf("WHY: %s\n\n", sprintf(why, X_LAB)))
}

# X_COL is resolved in the load section, once the object exists:
#     X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis score')
# Every downstream cell reads X_COL, never a literal score column name.

---
## 02 · Coarse clusters and root selection

**Why.** Trajectory inference is fitted on fewer, larger clusters than the
resolution-0.6 annotation. Fine clusters fragment a lineage into steps the
algorithm then has to connect, which adds branches that reflect clustering
granularity rather than biology.

**Choice.** Resolution 0.2. The root is the most homeostatic cluster, read off
the homeostatic panel — a quantity independent of what the trajectory is being
used to test.

**Display.** Resolution-0.2 clusters and the scores on the UMAP, so the root
call is visible rather than asserted.

In [ ]:
# coarser clustering for trajectory (fewer, larger clusters)
DefaultAssay(mg) <- "RNA"
mg <- FindClusters(mg, resolution = 0.2, graph.name = "RNA_snn")   # try 0.2–0.3
cat("clusters at res 0.2:\n"); print(table(mg$seurat_clusters))
# inspect sizes — want ~5-8 clusters, none tiny (<100)

In [ ]:
# VIEW res-0.2 clusters + scores on UMAP → pick homeostatic root
suppressPackageStartupMessages({ library(Seurat); library(RColorBrewer); library(dplyr) })

CLUSTER_LBL <- "seurat_clusters"        # the new 7-cluster res-0.2 result
dimred <- Embeddings(mg, "umap.mg")[, 1:2]
clus   <- factor(as.character(mg@meta.data[[CLUSTER_LBL]]))
pal    <- brewer.pal(max(3, nlevels(clus)), "Set1")

# helper: quantile-binned diverging color for a score
score_col <- function(v) {
    pal2 <- colorRampPalette(c("#2166AC","#92C5DE","grey88","#F4A582","#B2182B"))(100)
    pal2[cut(v, quantile(v, seq(0,1,length.out=101), na.rm=TRUE), include.lowest=TRUE, labels=FALSE)]
}

png("mg_res02_clusters_and_scores.png", width=1800, height=500, res=130)
par(mfrow=c(1,4), mar=c(4,4,3,1))

# panel 1: clusters with centroid labels
plot(dimred, col=pal[clus], cex=0.35, pch=16, asp=1, xlab="UMAP1", ylab="UMAP2", main="Clusters (res 0.2)")
for (i in levels(clus)) { ctr <- colMeans(dimred[clus==i,,drop=FALSE]); 
    points(ctr[1],ctr[2],pch=21,bg="white",cex=2.5,lwd=1.5); text(ctr[1],ctr[2],i,font=2,cex=1) }

# panels 2-4: scores
plot(dimred, col=score_col(mg$Score_Homeostatic), cex=0.35, pch=16, asp=1,
     xlab="UMAP1", ylab="", main="Homeostatic (red=high → root here)")
plot(dimred, col=score_col(mg$DAM_Homeo_axis_z), cex=0.35, pch=16, asp=1,
     xlab="UMAP1", ylab="", main="DAM activation (blue=low → root here)")
plot(dimred, col=score_col(mg$sen_score), cex=0.35, pch=16, asp=1,
     xlab="UMAP1", ylab="", main="Senescence (should look flat)")
dev.off()
cat("saved: mg_res02_clusters_and_scores.png\n")

# numeric confirmation: per-cluster scores
DefaultAssay(mg) <- "RNA"
hm <- intersect(c("P2RY12","CX3CR1","TMEM119","SELPLG","CSF1R"), rownames(mg))
mg$homeo_expr <- colMeans(GetAssayData(mg, layer="data")[hm,,drop=FALSE])
tab <- mg@meta.data %>% group_by(.data[[CLUSTER_LBL]]) %>%
    summarise(n=n(), homeo_marker=mean(homeo_expr), homeo_score=mean(Score_Homeostatic),
              dam=mean(DAM_Homeo_axis_z), sen=mean(sen_score), .groups="drop") %>%
    arrange(desc(homeo_marker))
cat("\n════ per-cluster scores (top = most homeostatic = root candidate) ════\n")
print(as.data.frame(tab), digits=3)

---
## 03 · Slingshot fit

**Why.** Slingshot fits a minimum spanning tree over cluster centroids, then
principal curves through the cells. Fitting on centroids rather than cells is
what makes the topology reproducible.

**Inputs.** The reduction and the resolution-0.2 cluster labels. Nothing else —
no score, no group label, no senescence call.

**Seeding note.** The source clustering is unseeded but reproducible when run
in order from a fresh kernel. Adding `set.seed()` changes the RNG state and
yields a different clustering, so the seed is deliberately absent.

In [ ]:
# RE-FIT at res-0.2 clusters (7), root = cluster 3 (most homeostatic)
suppressPackageStartupMessages({ library(slingshot); library(SingleCellExperiment) })

ROOT_CLUS   <- "3"
CLUSTER_LBL <- "seurat_clusters"

sce <- as.SingleCellExperiment(mg)
stopifnot("HARMONY" %in% reducedDimNames(sce))
stopifnot(ROOT_CLUS %in% as.character(mg@meta.data[[CLUSTER_LBL]]))

sce <- slingshot(sce,
                 clusterLabels = as.character(mg@meta.data[[CLUSTER_LBL]]),
                 reducedDim    = reducedDim(sce, "HARMONY")[, 1:30],
                 start.clus    = ROOT_CLUS)

sds  <- SlingshotDataSet(sce)
lins <- slingLineages(sds)
cat("✓ re-fit | root =", ROOT_CLUS, "| n lineages =", length(lins), "\n\n")
for (i in seq_along(lins)) cat(sprintf("Lineage%d: %s\n", i, paste(lins[[i]], collapse=" → ")))
mg$pseudotime <- rowMeans(slingPseudotime(sce), na.rm=TRUE)

# immediate monotonicity check (confirm artifacts are gone)
pt_mat <- slingPseudotime(sce); cl <- as.character(mg@meta.data[[CLUSTER_LBL]])
cat("\n════ per-cluster pseudotime per lineage ════\n")
for (i in seq_along(lins)) {
    path <- lins[[i]]; pti <- pt_mat[,i]
    means <- sapply(path, function(c) mean(pti[cl==c], na.rm=TRUE))
    cat(sprintf("L%d: %s  | monotonic: %s\n", i, paste(sprintf("%s(%.1f)",path,means), collapse=" → "),
                ifelse(all(diff(means) > -1, na.rm=TRUE), "YES", "NO")))
}

---
## 04 · Lineage validity filter

**Why.** Slingshot returns every lineage the MST supports, including artifacts
— lineages that double back, or that consist of a handful of cells. Keeping
only monotonic progressions is what stops a spurious branch from carrying a
correlation into section 07.

**Display.** Each lineage with its cell count and monotonicity verdict, kept or
dropped, with the reason.

In [ ]:
# FILTER LINEAGES — keep only monotonic (valid) progressions, drop artifacts
suppressPackageStartupMessages({ library(slingshot); library(SingleCellExperiment) })

pt_mat <- slingPseudotime(sce)
clus   <- as.character(OBJ@meta.data[[CLUSTER_LBL]])

# tolerance: allow tiny non-monotonic dips (noise), drop only real violations
TOL <- 1.0     # a step may dip by up to TOL pseudotime units without being flagged

lineage_status <- data.frame(lineage=integer(), path=character(),
                             monotonic=logical(), max_drop=numeric(),
                             min_n=integer(), stringsAsFactors=FALSE)

for (i in seq_along(lins)) {
    path <- lins[[i]]
    pti  <- pt_mat[, i]
    means <- sapply(path, function(cl) mean(pti[clus == cl], na.rm=TRUE))
    ns    <- sapply(path, function(cl) sum(clus == cl & is.finite(pti)))
    drops <- diff(means)                          # negative = pseudotime goes backwards
    max_drop <- if (length(drops)) min(drops, na.rm=TRUE) else 0
    is_mono  <- max_drop >= -TOL                  # valid if no drop worse than TOL
    lineage_status <- rbind(lineage_status, data.frame(
        lineage=i, path=paste(path, collapse="→"),
        monotonic=is_mono, max_drop=round(max_drop,1), min_n=min(ns)))
}

cat("════ lineage validity (monotonic ascending pseudotime?) ════\n")
print(lineage_status, row.names=FALSE)

keep_lin <- lineage_status$lineage[lineage_status$monotonic]
drop_lin <- lineage_status$lineage[!lineage_status$monotonic]
cat("\n✓ KEEP (valid progressions):", paste0("L", keep_lin, collapse=", "), "\n")
cat("✗ DROP (non-monotonic, likely MST artifacts):", paste0("L", drop_lin, collapse=", "), "\n")

# the filtered lineage set for downstream plots/analysis
lins_kept <- lins[keep_lin]

---
## 05 · Branching tree

**Why.** The topology on its own, before any score is laid over it.

In [ ]:
# FIG A — BRANCHING TREE (4 lineages, res-0.2, root cl.3)
suppressPackageStartupMessages({ library(Seurat); library(slingshot); library(SingleCellExperiment); library(RColorBrewer) })
ROOT_CLUS<-"3"; CLUSTER_LBL<-"seurat_clusters"; DISP_RED<-"umap.mg"

dimred <- Embeddings(mg, DISP_RED)[,1:2]
clus   <- factor(as.character(mg@meta.data[[CLUSTER_LBL]]))
pal    <- brewer.pal(max(3,nlevels(clus)),"Set1")
clus_ids <- levels(clus)
sds <- SlingshotDataSet(sce); lins <- slingLineages(sds)

# branch points / leaves
nx <- list(); for (L in lins) if(length(L)>=2) for(k in seq_len(length(L)-1)) nx[[L[k]]]<-union(nx[[L[k]]],L[k+1])
bp <- names(nx)[sapply(nx,length)>1]; lv <- setdiff(unlist(lapply(lins,tail,1)), bp)

png("mg_tree_res02_root3.png", width=1600, height=800, res=130)
par(mfrow=c(1,2), mar=c(4,4,2.5,1))
plot(dimred, col=pal[clus], cex=0.35, pch=16, asp=1, xlab="UMAP1", ylab="UMAP2", main="Clusters (res 0.2)")
for(i in clus_ids){ctr<-colMeans(dimred[clus==i,,drop=FALSE]);text(ctr[1],ctr[2],i,font=2,cex=1.2)}
plot(dimred, col=pal[clus], cex=0.35, pch=16, asp=1, xlab="UMAP1", ylab="UMAP2",
     main=sprintf("Lineage tree (root cl.%s) — %d lineages", ROOT_CLUS, length(lins)))
lines(sds, type='lineages', lwd=2.5, col='black')
for(i in clus_ids){ctr<-colMeans(dimred[clus==i,,drop=FALSE])
    bg<-if(i==ROOT_CLUS)"#E4002B" else if(i%in%bp)"#F5A623" else if(i%in%lv)"#2E86C1" else "white"
    points(ctr[1],ctr[2],pch=21,bg=bg,col="black",cex=2.8,lwd=1.5);text(ctr[1],ctr[2],i,font=2,cex=0.9)}
legend("topright",legend=c("root","branch point","leaf","pass-through"),
       pt.bg=c("#E4002B","#F5A623","#2E86C1","white"),pch=21,pt.cex=1.6,cex=0.8,bty="n")
dev.off(); cat("✓ mg_tree_res02_root3.png\n")

---
## 06 · Per-lineage UMAP coloured by score

**Why.** One facet per surviving lineage, coloured by score.

**Scores are used for visualization only.** They played no part in the fit;
this panel shows where they land on a trajectory that was built without them.

In [ ]:
# FIG B — FACETED per-lineage colored by SCORE (run for DAM, then sen)
SCORE<-"DAM_Homeo_axis_z"; SCORE_NAME<-"DAM-Homeo (activation)"   # then sen_score / "Senescence"

dimred <- Embeddings(mg, DISP_RED)[,1:2]
clus   <- factor(as.character(mg@meta.data[[CLUSTER_LBL]]))
cent   <- t(sapply(levels(clus), function(i) colMeans(dimred[clus==i,,drop=FALSE]))); rownames(cent)<-levels(clus)
root_xy<- cent[ROOT_CLUS,]
v <- mg@meta.data[[SCORE]]
pal2 <- colorRampPalette(c("#2166AC","#92C5DE","grey88","#F4A582","#B2182B"))(100)
ptcol <- pal2[cut(v, quantile(v,seq(0,1,length.out=101),na.rm=TRUE), include.lowest=TRUE, labels=FALSE)]

nlin<-length(lins); nc<-ceiling(sqrt(nlin)); nr<-ceiling(nlin/nc)
png(sprintf("mg_lin_res02_%s_root3.png", make.names(SCORE)), width=nc*520, height=nr*480, res=120)
par(mfrow=c(nr,nc), mar=c(3,3,2.6,1), oma=c(0,0,2.4,0))
for(i in seq_len(nlin)){
    path<-lins[[i]]
    plot(dimred, col=ptcol, cex=0.35, pch=16, asp=1, xlab="", ylab="",
         main=sprintf("Lineage%d: %s", i, paste(path,collapse="→")), cex.main=0.85)
    pc<-cent[path,,drop=FALSE]; lines(pc,lwd=3,col="black")
    points(pc,pch=21,bg="white",col="black",cex=2.2,lwd=1.3); text(pc[,1],pc[,2],path,font=2,cex=0.75)
    points(root_xy[1],root_xy[2],pch=21,bg="black",col="white",cex=2.4,lwd=1.5)
}
mtext(sprintf("Lineages colored by %s (root cl.%s)", SCORE_NAME, ROOT_CLUS), outer=TRUE, cex=1.05, font=2)
dev.off(); cat(sprintf("✓ mg_lin_res02_%s_root3.png\n", make.names(SCORE)))

---
## 07 · Score ~ pseudotime correlation per lineage

**Why.** The directional claim, and the only place the trajectory is used
inferentially. Per lineage, Spearman between each raw score and pseudotime.

**Raw scores.** Not the composite axes — see the note at the top of this
notebook. A composite would correlate with pseudotime partly through its shared
`−Homeostatic` term, and the root is at the homeostatic end, so that term is
itself a function of pseudotime.

**Display.** Per-lineage rho with CI, one row per score.

In [ ]:
# FIG C — sen~pseudotime correlation per lineage (orthogonality confirmation)
pt_mat <- slingPseudotime(sce); md <- mg@meta.data
cat("════ sen~pseudotime & DAM~pseudotime per lineage (Spearman) ════\n")
res <- data.frame()
for(i in seq_len(ncol(pt_mat))){
    keep <- is.finite(pt_mat[,i])
    rs <- cor(md$sen_score[keep], pt_mat[keep,i], method="spearman")
    rd <- cor(md$DAM_Homeo_axis_z[keep], pt_mat[keep,i], method="spearman")
    res <- rbind(res, data.frame(lineage=paste0("L",i), n=sum(keep),
                                 rho_sen=round(rs,3), rho_dam=round(rd,3)))
}
print(res, row.names=FALSE)
cat("\n>>> sen ≈ 0 (flat), dam strongly positive ⇒ trajectory tracks DAM, not senescence\n")

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# per-lineage correlations: sen / DAM / IFN(IRM) ~ pseudotime
#   does the trajectory bifurcate into DAM vs interferon branches?
pt_mat <- slingPseudotime(sce); md <- mg@meta.data
stopifnot(AXIS_SPEC$cols[1] %in% colnames(md))

cat("════ score ~ pseudotime per lineage (Spearman) ════\n")
res <- data.frame()
for(i in seq_len(ncol(pt_mat))){
    k  <- is.finite(pt_mat[,i]); p <- pt_mat[k,i]
    res <- rbind(res, data.frame(
        lineage = paste0("L",i), n = sum(k),
        rho_sen = round(cor(md$sen_score[k],        p, method="spearman"),3),
        rho_dam = round(cor(md$DAM_Homeo_axis_z[k], p, method="spearman"),3),
        rho_ifn = round(cor(md$Score_IRM[k],        p, method="spearman"),3)))
}
print(res, row.names=FALSE)
cat("\nlineage paths:\n")
for(i in seq_along(lins)) cat(sprintf("  L%d: %s\n", i, paste(lins[[i]], collapse=" → ")))

# also per-cluster IFN means, to see which leaf is the IFN endpoint
cl <- as.character(md[["seurat_clusters"]])
cat("\n════ per-cluster mean scores (which clusters are DAM vs IFN high?) ════\n")
suppressPackageStartupMessages(library(dplyr))
ct <- md %>% group_by(seurat_clusters) %>%
    summarise(n=n(), dam=mean(DAM_Homeo_axis_z), ifn=mean(Score_IRM),
              sen=mean(sen_score), .groups="drop") %>% arrange(desc(ifn))
print(as.data.frame(ct), digits=3)

---
## 08 · Cell density along pseudotime

**Why.** Whether the groups occupy the same trajectory at different densities,
or different parts of it. Both are compatible with the same mean pseudotime, so
the mean alone does not distinguish them.

**Display.** Density along the shared trajectory, split by group.

In [ ]:
# CELL DENSITY ALONG PSEUDOTIME — split by group (shared trajectory)
#   asks: do AD cells pile toward the activated end vs Control?
suppressPackageStartupMessages({ library(ggplot2); library(dplyr) })

gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
df <- mg_s@meta.data
df$pseudotime <- mg_s$pseudotime
df$grp2 <- factor(unname(gmap[as.character(df$Study_Group)]), levels=c("Control","AD"))
df <- df[!is.na(df$grp2) & is.finite(df$pseudotime), ]

RED <- "#C0392B"; BLUE <- "#3D7C9A"

# ---- DENSITY PLOT ----------------------------------------------------------
p_dens <- ggplot(df, aes(pseudotime, fill=grp2, color=grp2)) +
    geom_density(alpha=0.30, linewidth=0.9, adjust=1.2) +
    scale_fill_manual(values=c(Control=BLUE, AD=RED), name=NULL) +
    scale_color_manual(values=c(Control=BLUE, AD=RED), name=NULL) +
    labs(title="Cell density along the activation trajectory, by group",
         subtitle="Shared pseudotime · resting (low) → activated (high) · shift right = more activated cells",
         x="pseudotime  (resting → activated)", y="cell density") +
    theme_classic(base_size=11) +
    theme(plot.title=element_text(size=11, face="bold"),
          plot.subtitle=element_text(size=8, color="grey45"),
          legend.position="top",
          panel.border=element_rect(color="black", fill=NA, linewidth=0.4))
options(repr.plot.width=7, repr.plot.height=4.5); print(p_dens)
if (exists("save_figure")) save_figure(p_dens, "mg_pseudotime_density_byGroup", width=7, height=4.5)

# ---- HONEST TEST: is the group shift real at the DONOR level? ---------------
# each donor's MEDIAN pseudotime; compare AD vs Control across donors (not cells)
donor_pt <- df %>% group_by(Donor, grp2) %>%
    summarise(median_pt = median(pseudotime), n_cells = n(), .groups="drop") %>%
    filter(n_cells >= 20)                       # donors with enough cells
cat("\n════ donors per group (≥20 cells) ════\n"); print(table(donor_pt$grp2))
cat("\n════ donor-level median pseudotime, AD vs Control ════\n")
print(donor_pt %>% group_by(grp2) %>%
        summarise(med_of_medians = median(median_pt), IQR = IQR(median_pt)))
cat("\nWilcoxon (donor-level):\n")
print(wilcox.test(median_pt ~ grp2, data = donor_pt))

# fraction of each group's cells in the "activated" half (above global median)
gm <- median(df$pseudotime)
cat(sprintf("\n════ %% cells past global median (%.1f) = 'activated half' ════\n", gm))
print(df %>% group_by(grp2) %>% summarise(pct_activated = round(100*mean(pseudotime > gm),1)))

cat("\n✓ density-by-group done.\n")

---
## 09 · Progression panels

**Why.** Donor-ordered traces — activation, senescence, SASP, %SnC and the
clinical score — on one axis, so the order in which they rise can be read.

**Ordering.** Group, then Braak, then CDR. This is a *donor* ordering, not a
fitted trajectory, and it is descriptive: it shows which traces peak earlier
along a clinical progression, it does not establish that one causes another.

In [ ]:
# PROGRESSION PANEL (fresh) — NA→0 on clinical scores, sort group → Braak → CDR
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
`%||%` <- function(a, b) if (is.null(a)) b else a
na_fb  <- function(x, fb) { x[is.na(x)] <- fb; x }

# ─── (1) NA → 0 on clinical/pathology scores ───
PATH_COLS <- intersect(c("Braak","CDR"), colnames(prog))
for (c_ in PATH_COLS) {
    n_na <- sum(is.na(prog[[c_]]))
    prog[[c_]][is.na(prog[[c_]])] <- 0
    cat(sprintf("  %s: %d NA → 0\n", c_, n_na))
}

# ─── (2) sort: group → Braak → CDR (no NAs now) ───
SORT_PATH <- "Braak"; SORT_TIE <- "CDR"
plot_df <- prog
plot_df$Study_Group <- factor(plot_df$Study_Group, levels = sg_ordered)
plot_df <- plot_df[order(plot_df$Study_Group, plot_df[[SORT_PATH]], plot_df[[SORT_TIE]]), ]
plot_df$x <- seq_len(nrow(plot_df)); rownames(plot_df) <- NULL
cat(sprintf("  sorted Study_Group → %s → %s | %d donors\n", SORT_PATH, SORT_TIE, nrow(plot_df)))

# ─── (3) z-scores + smoothed traces ───
z_score <- function(v){ v<-as.numeric(v); if(sum(!is.na(v))==0) return(v)
    s<-sd(v,na.rm=TRUE); if(s==0) return(v-mean(v,na.rm=TRUE)); (v-mean(v,na.rm=TRUE))/s }
smooth_trace <- function(y, wf=0.10, mn=5, mx=21){ y<-as.numeric(y); n<-length(y)
    if(n<3||sum(!is.na(y))<3) return(y); w<-max(mn,min(mx,round(n*wf))); if(w%%2==0) w<-w+1
    h<-(w-1)%/%2; out<-numeric(n); for(i in seq_len(n)){lo<-max(1,i-h);hi<-min(n,i+h);out[i]<-mean(y[lo:hi],na.rm=TRUE)}
    out[is.nan(out)]<-NA; out }

modules_in_prog   <- intersect(PROGRESSION_MODULES, gsub("^mean_","", grep("^mean_", colnames(prog), value=TRUE)))
available_markers <- CANONICAL_MARKER_ORDER[CANONICAL_MARKER_ORDER %in% colnames(prog)]

plot_df$pct_sen_z <- z_score(plot_df$pct_sen)
for (mod in modules_in_prog) plot_df[[paste0("mean_",mod,"_z")]] <- z_score(plot_df[[paste0("mean_",mod)]])
for (m in available_markers)  plot_df[[paste0(m,"_z")]]          <- z_score(plot_df[[m]])

trace_rows <- list()
for (m in available_markers) trace_rows[[paste0("p_",m)]] <- data.frame(
    x=plot_df$x, y=smooth_trace(plot_df[[paste0(m,"_z")]]), trace=MARKER_LABELS[m],
    color=MARKER_COLORS[m], linetype=ifelse(m %in% INVERSE_MARKERS,"dashed","solid"),
    size=0.8, group="pathology", legord=match(m,CANONICAL_MARKER_ORDER), stringsAsFactors=FALSE)
for (mod in modules_in_prog) trace_rows[[paste0("m_",mod)]] <- data.frame(
    x=plot_df$x, y=smooth_trace(plot_df[[paste0("mean_",mod,"_z")]]), trace=mod,
    color=na_fb(MODULE_TRACE_COLORS[mod], fallback_color), linetype="solid",
    size=1.1, group="module", legord=100+match(mod,modules_in_prog), stringsAsFactors=FALSE)
y_snc <- smooth_trace(plot_df$pct_sen_z)
trace_rows[["snc"]] <- data.frame(x=plot_df$x, y=y_snc, trace="%SnC",
    color=MODULE_TRACE_COLORS[["%SnC"]], linetype="solid", size=1.6, group="snc", legord=999, stringsAsFactors=FALSE)
trace_df <- do.call(rbind, trace_rows); rownames(trace_df) <- NULL
trace_df$trace <- factor(trace_df$trace, levels=unique(trace_df$trace[order(trace_df$legord)]))

# ─── (4) bands / labels / dividers / dots ───
band_rows<-list(); label_rows<-list(); divider_x<-numeric(0)
for (sg in sg_ordered){ d<-plot_df[plot_df$Study_Group==sg,]; if(nrow(d)==0) next
    x0<-min(d$x)-0.5; x1<-max(d$x)+0.5; bc<-na_fb(STUDY_GROUP_PALETTE[sg], fallback_color)
    band_rows[[sg]]<-data.frame(xmin=x0,xmax=x1,ymin=-Inf,ymax=Inf,fill=bc,stringsAsFactors=FALSE)
    label_rows[[sg]]<-data.frame(x=(x0+x1)/2,label_main=sg,label_n=sprintf("n=%d",nrow(d)),color=bc,stringsAsFactors=FALSE)}
band_df<-do.call(rbind,band_rows); label_df<-do.call(rbind,label_rows)
for (i in seq_along(sg_ordered)[-length(sg_ordered)]){ d<-plot_df[plot_df$Study_Group==sg_ordered[i],]
    if(nrow(d)>0) divider_x<-c(divider_x,max(d$x)+0.5)}
dot_df<-plot_df[,c("x","Study_Group")]; dot_df$y<-y_snc
dot_df$color<-na_fb(STUDY_GROUP_PALETTE[as.character(dot_df$Study_Group)], fallback_color)
dot_df<-dot_df[!is.na(dot_df$y),]

y_range<-range(trace_df$y,na.rm=TRUE); y_pad<-0.12*diff(y_range); y_lim<-c(y_range[1]-y_pad,y_range[2]+y_pad)

# ─── (5) main panel ───
p_overall <- ggplot() +
    geom_rect(data=band_df, aes(xmin=xmin,xmax=xmax,ymin=ymin,ymax=ymax,fill=fill), alpha=0.12, inherit.aes=FALSE) +
    scale_fill_identity() +
    {if(length(divider_x)>0) geom_vline(xintercept=divider_x, color="grey60", linetype="dotted", linewidth=0.4)} +
    geom_hline(yintercept=0, color="grey60", linetype="dashed", linewidth=0.4) +
    geom_line(data=trace_df[trace_df$group %in% c("pathology","module"),], aes(x=x,y=y,group=trace),
              color="white", linewidth=2.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[trace_df$group %in% c("pathology","module"),],
              aes(x=x,y=y,color=color,linetype=linetype,group=trace,linewidth=size)) +
    geom_line(data=trace_df[trace_df$group=="snc",], aes(x=x,y=y), color="white", linewidth=3.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[trace_df$group=="snc",], aes(x=x,y=y), color=MODULE_TRACE_COLORS[["%SnC"]], linewidth=1.8, lineend="round") +
    geom_point(data=dot_df, aes(x=x,y=y,fill=color), shape=21, color="black", size=1.7, stroke=0.3) +
    scale_color_identity()+scale_linetype_identity()+scale_linewidth_identity() +
    scale_x_continuous(expand=c(0,0)) + scale_y_continuous(limits=y_lim) +
    labs(title=sprintf("Progression: %s | %s | group → Braak → CDR | n=%d", CELL_TYPE, DATASET, nrow(plot_df)),
         x="Donors (group → Braak → CDR)", y="z-score") +
    theme_minimal(base_size=10) +
    theme(plot.title=element_text(size=11,face="bold",color="#333333",hjust=0,margin=margin(b=6)),
          plot.title.position="plot", plot.background=element_rect(fill="white",color=NA),
          panel.border=element_rect(color="#333333",fill=NA,linewidth=0.6),
          panel.grid.major.y=element_line(color="grey92",linewidth=0.3), panel.grid.minor=element_blank(),
          panel.grid.major.x=element_blank(), axis.text.x=element_blank(), axis.ticks.x=element_blank(),
          axis.title.x=element_text(size=8.5,color="#666666",margin=margin(t=6)),
          axis.text.y=element_text(size=9,color="#333333"), axis.title.y=element_text(size=9.5,color="#333333"),
          legend.position="none", plot.margin=margin(8,8,4,8))

# ─── (6) strip + legend ───
strip_plot <- ggplot(label_df) +
    geom_text(aes(x=x,y=1,label=label_main,color=color), size=2.7, fontface="bold", vjust=1) +
    geom_text(aes(x=x,y=0.45,label=label_n,color=color), size=2.4, vjust=1) +
    scale_color_identity() + scale_x_continuous(limits=c(0.5,nrow(plot_df)+0.5),expand=c(0,0)) +
    scale_y_continuous(limits=c(0,1.1)) + theme_void() +
    theme(plot.margin=margin(2,8,0,8), plot.background=element_rect(fill="white",color=NA))

legend_items<-list()
for (m in available_markers) legend_items[[m]]<-data.frame(label=MARKER_LABELS[m], color=MARKER_COLORS[m],
    linetype=ifelse(m %in% INVERSE_MARKERS,"dashed","solid"), bold=FALSE, stringsAsFactors=FALSE)
for (mod in modules_in_prog) legend_items[[mod]]<-data.frame(label=mod, color=na_fb(MODULE_TRACE_COLORS[mod],fallback_color),
    linetype="solid", bold=FALSE, stringsAsFactors=FALSE)
legend_items[["snc"]]<-data.frame(label="%SnC", color=MODULE_TRACE_COLORS[["%SnC"]], linetype="solid", bold=TRUE, stringsAsFactors=FALSE)
leg_df<-do.call(rbind,legend_items); rownames(leg_df)<-NULL
n_items<-nrow(leg_df); leg_df$slot<-seq_len(n_items)
leg_df$x_seg_lo<-leg_df$slot-0.5+0.05; leg_df$x_seg_hi<-leg_df$x_seg_lo+0.30; leg_df$x_text<-leg_df$x_seg_hi+0.05
legend_plot <- ggplot(leg_df) +
    geom_segment(aes(x=x_seg_lo,xend=x_seg_hi,y=1,yend=1,color=color,linetype=linetype), linewidth=0.9, lineend="round") +
    geom_text(aes(x=x_text,y=1,label=label,color=color,fontface=ifelse(bold,"bold","plain")), size=2.7, hjust=0, vjust=0.5) +
    scale_color_identity()+scale_linetype_identity() +
    scale_x_continuous(limits=c(0.4,n_items+0.6),expand=c(0,0)) + scale_y_continuous(limits=c(0.5,1.5),expand=c(0,0)) +
    theme_void() + theme(plot.margin=margin(2,8,4,8), plot.background=element_rect(fill="white",color=NA))

# ─── (7) compose + save ───
composed <- strip_plot / p_overall / legend_plot + plot_layout(heights=c(0.10,1,0.08))
save_figure(composed, slug=sprintf("%s_progression_braak_na0", CELL_TYPE), width=9.5, height=4.6)
options(repr.plot.width=10, repr.plot.height=4.9); print(composed)

# ─── (8) peak diagnostics ───
cat("\nsmoothed trace peaks:\n")
rp<-function(nm,v){sm<-smooth_trace(v); if(all(is.na(sm))){cat(sprintf("  %-18s all NaN\n",nm));return()}
    pk<-which.max(sm); cat(sprintf("  %-18s peak idx=%3d group=%s z=%+.2f\n",nm,pk,as.character(plot_df$Study_Group[pk]),sm[pk]))}
rp("%SnC", plot_df$pct_sen_z); for(mod in modules_in_prog) rp(mod, plot_df[[paste0("mean_",mod,"_z")]])

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# PROGRESSION — 5 traces: IFN-z, DAM-z, SASP-z, %SnC, CDR-z | group→Braak→CDR
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
`%||%` <- function(a,b) if (is.null(a)) b else a
na_fb  <- function(x,fb){ x[is.na(x)]<-fb; x }
z_score <- function(v){ v<-as.numeric(v); if(sum(!is.na(v))==0) return(v)
    s<-sd(v,na.rm=TRUE); if(s==0) return(v-mean(v,na.rm=TRUE)); (v-mean(v,na.rm=TRUE))/s }
smooth_trace <- function(y,wf=0.10,mn=5,mx=21){ y<-as.numeric(y); n<-length(y)
    if(n<3||sum(!is.na(y))<3) return(y); w<-max(mn,min(mx,round(n*wf))); if(w%%2==0) w<-w+1
    h<-(w-1)%/%2; out<-numeric(n); for(i in seq_len(n)){lo<-max(1,i-h);hi<-min(n,i+h);out[i]<-mean(y[lo:hi],na.rm=TRUE)}
    out[is.nan(out)]<-NA; out }

# ─── NA→0 on clinical scores, then sort ───
for (c_ in intersect(c("Braak","CDR"), colnames(prog))) prog[[c_]][is.na(prog[[c_]])] <- 0
plot_df <- prog
plot_df$Study_Group <- factor(plot_df$Study_Group, levels=sg_ordered)
plot_df <- plot_df[order(plot_df$Study_Group, plot_df$Braak, plot_df$CDR), ]
plot_df$x <- seq_len(nrow(plot_df)); rownames(plot_df) <- NULL
cat(sprintf("sorted | %d donors\n", nrow(plot_df)))

# ─── define the 5 traces: column, label, color, heavy? ───
TRACES <- list(
    list(col="mean_Homeo", lab="Homeostatic", color="#17BECF", size=1.1, heavy=FALSE),  # cyan
    list(col="mean_IFN",   lab="IFN (IRM)",   color="#1F77B4", size=1.1, heavy=FALSE),  # blue
    list(col="mean_DAM",   lab="DAM",         color="#D62728", size=1.1, heavy=FALSE),  # red
    list(col="mean_SASP",  lab="SASP",        color="#FF7F0E", size=1.1, heavy=FALSE),  # orange
    list(col="CDR",        lab="CDR (clin.)", color="#2CA02C", size=1.1, heavy=FALSE),  # green
    list(col="pct_sen",    lab="%SnC",        color="#6A0DAD", size=1.8, heavy=TRUE)     # purple (heavy)
)
for (t in TRACES) stopifnot(t$col %in% colnames(plot_df))
for (t in TRACES) stopifnot(t$col %in% colnames(plot_df))

# z-score + smooth each
trace_rows <- list()
for (t in TRACES) {
    z <- z_score(plot_df[[t$col]])
    trace_rows[[t$col]] <- data.frame(
        x=plot_df$x, y=smooth_trace(z), trace=t$lab, color=t$color,
        size=t$size, heavy=t$heavy, stringsAsFactors=FALSE)
}
trace_df <- do.call(rbind, trace_rows); rownames(trace_df) <- NULL
trace_df$trace <- factor(trace_df$trace, levels=sapply(TRACES, `[[`, "lab"))
y_snc <- trace_rows[["pct_sen"]]$y

# ─── bands / labels / dividers / dots ───
band_rows<-list(); label_rows<-list(); divider_x<-numeric(0)
for (sg in sg_ordered){ d<-plot_df[plot_df$Study_Group==sg,]; if(nrow(d)==0) next
    x0<-min(d$x)-0.5; x1<-max(d$x)+0.5; bc<-na_fb(STUDY_GROUP_PALETTE[sg],fallback_color)
    band_rows[[sg]]<-data.frame(xmin=x0,xmax=x1,ymin=-Inf,ymax=Inf,fill=bc,stringsAsFactors=FALSE)
    label_rows[[sg]]<-data.frame(x=(x0+x1)/2,label_main=sg,label_n=sprintf("n=%d",nrow(d)),color=bc,stringsAsFactors=FALSE)}
band_df<-do.call(rbind,band_rows); label_df<-do.call(rbind,label_rows)
for (i in seq_along(sg_ordered)[-length(sg_ordered)]){ d<-plot_df[plot_df$Study_Group==sg_ordered[i],]
    if(nrow(d)>0) divider_x<-c(divider_x,max(d$x)+0.5)}
dot_df<-data.frame(x=plot_df$x, y=y_snc,
    color=na_fb(STUDY_GROUP_PALETTE[as.character(plot_df$Study_Group)],fallback_color))
dot_df<-dot_df[!is.na(dot_df$y),]
y_range<-range(trace_df$y,na.rm=TRUE); y_pad<-0.12*diff(y_range); y_lim<-c(y_range[1]-y_pad,y_range[2]+y_pad)

# ─── main panel ───
p_overall <- ggplot() +
    geom_rect(data=band_df, aes(xmin=xmin,xmax=xmax,ymin=ymin,ymax=ymax,fill=fill), alpha=0.12, inherit.aes=FALSE) +
    scale_fill_identity() +
    {if(length(divider_x)>0) geom_vline(xintercept=divider_x, color="grey60", linetype="dotted", linewidth=0.4)} +
    geom_hline(yintercept=0, color="grey60", linetype="dashed", linewidth=0.4) +
    # non-heavy traces + white halo
    geom_line(data=trace_df[!trace_df$heavy,], aes(x=x,y=y,group=trace), color="white", linewidth=2.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[!trace_df$heavy,], aes(x=x,y=y,color=color,group=trace,linewidth=size)) +
    # %SnC heavy + halo
    geom_line(data=trace_df[trace_df$heavy,], aes(x=x,y=y), color="white", linewidth=3.2, alpha=0.85, lineend="round") +
    geom_line(data=trace_df[trace_df$heavy,], aes(x=x,y=y,color=color), linewidth=1.8, lineend="round") +
    geom_point(data=dot_df, aes(x=x,y=y,fill=color), shape=21, color="black", size=1.7, stroke=0.3) +
    scale_color_identity()+scale_linewidth_identity() +
    scale_x_continuous(expand=c(0,0)) + scale_y_continuous(limits=y_lim) +
    labs(title=sprintf("Progression: %s | %s | group → Braak → CDR | n=%d", CELL_TYPE, DATASET, nrow(plot_df)),
         x="Donors (group → Braak → CDR)", y="z-score") +
    theme_minimal(base_size=10) +
    theme(plot.title=element_text(size=11,face="bold",color="#333333",hjust=0,margin=margin(b=6)),
          plot.title.position="plot", plot.background=element_rect(fill="white",color=NA),
          panel.border=element_rect(color="#333333",fill=NA,linewidth=0.6),
          panel.grid.major.y=element_line(color="grey92",linewidth=0.3), panel.grid.minor=element_blank(),
          panel.grid.major.x=element_blank(), axis.text.x=element_blank(), axis.ticks.x=element_blank(),
          axis.title.x=element_text(size=8.5,color="#666666",margin=margin(t=6)),
          axis.text.y=element_text(size=9,color="#333333"), axis.title.y=element_text(size=9.5,color="#333333"),
          legend.position="none", plot.margin=margin(8,8,4,8))

# ─── strip ───
strip_plot <- ggplot(label_df) +
    geom_text(aes(x=x,y=1,label=label_main,color=color), size=2.7, fontface="bold", vjust=1) +
    geom_text(aes(x=x,y=0.45,label=label_n,color=color), size=2.4, vjust=1) +
    scale_color_identity() + scale_x_continuous(limits=c(0.5,nrow(plot_df)+0.5),expand=c(0,0)) +
    scale_y_continuous(limits=c(0,1.1)) + theme_void() +
    theme(plot.margin=margin(2,8,0,8), plot.background=element_rect(fill="white",color=NA))

# ─── legend (5 items) ───
leg_df <- data.frame(label=sapply(TRACES,`[[`,"lab"), color=sapply(TRACES,`[[`,"color"),
                     bold=sapply(TRACES,`[[`,"heavy"), stringsAsFactors=FALSE)
n_items<-nrow(leg_df); leg_df$slot<-seq_len(n_items)
leg_df$x_seg_lo<-leg_df$slot-0.5+0.05; leg_df$x_seg_hi<-leg_df$x_seg_lo+0.30; leg_df$x_text<-leg_df$x_seg_hi+0.05
legend_plot <- ggplot(leg_df) +
    geom_segment(aes(x=x_seg_lo,xend=x_seg_hi,y=1,yend=1,color=color), linewidth=0.9, lineend="round") +
    geom_text(aes(x=x_text,y=1,label=label,color=color,fontface=ifelse(bold,"bold","plain")), size=2.7, hjust=0, vjust=0.5) +
    scale_color_identity() +
    scale_x_continuous(limits=c(0.4,n_items+0.6),expand=c(0,0)) + scale_y_continuous(limits=c(0.5,1.5),expand=c(0,0)) +
    theme_void() + theme(plot.margin=margin(2,8,4,8), plot.background=element_rect(fill="white",color=NA))

# ─── compose + save ───
composed <- strip_plot / p_overall / legend_plot + plot_layout(heights=c(0.10,1,0.08))
save_figure(composed, slug=sprintf("%s_progression_5trace", CELL_TYPE), width=9.5, height=4.6)
options(repr.plot.width=10, repr.plot.height=4.9); print(composed)

# ─── peaks ───
cat("\ntrace peaks:\n")
for (t in TRACES) { sm<-smooth_trace(z_score(plot_df[[t$col]])); pk<-which.max(sm)
    cat(sprintf("  %-12s peak idx=%3d group=%-20s z=%+.2f\n", t$lab, pk, as.character(plot_df$Study_Group[pk]), sm[pk])) }

In [ ]:
# <<< AXIS-CHECK: hardcoded token(s) ['IRM'] remain in this lifted
#     block — replace with X_LAB / AXIS_SPEC$cols[1] / QUAD_COL.
# PAIRED PROGRESSION PANELS — %SnC + CDR anchored, paired with each axis
#   publication quality · 2×2 panels · two-row legend
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })

z_score <- function(v){ v<-as.numeric(v); s<-sd(v,na.rm=TRUE)
    if(is.na(s)||s==0) return(v-mean(v,na.rm=TRUE)); (v-mean(v,na.rm=TRUE))/s }
smooth_trace <- function(y,wf=0.10,mn=5,mx=21){ y<-as.numeric(y); n<-length(y)
    if(n<3) return(y); w<-max(mn,min(mx,round(n*wf))); if(w%%2==0) w<-w+1; h<-(w-1)%/%2
    out<-sapply(seq_len(n),function(i) mean(y[max(1,i-h):min(n,i+h)],na.rm=TRUE)); out[is.nan(out)]<-NA; out }

# ─── NA→0 on clinical scores, sort group → Braak → CDR ───
for (c_ in intersect(c("Braak","CDR"), colnames(prog))) prog[[c_]][is.na(prog[[c_]])] <- 0
plot_df <- prog
plot_df$Study_Group <- factor(plot_df$Study_Group, levels=sg_ordered)
plot_df <- plot_df[order(plot_df$Study_Group, plot_df$Braak, plot_df$CDR), ]
plot_df$x <- seq_len(nrow(plot_df)); rownames(plot_df) <- NULL
cat(sprintf("sorted | %d donors\n", nrow(plot_df)))

# ─── anchors (drawn in every panel) ───
snc_y <- smooth_trace(z_score(plot_df$pct_sen))
cdr_y <- smooth_trace(z_score(plot_df$CDR))

# ─── variables paired against the anchors ───
PAIRS <- list(
    list(col="mean_Homeo", lab="Homeostatic", color="#17BECF"),
    list(col="mean_IFN",   lab="IFN (IRM)",   color="#1F77B4"),
    list(col="mean_DAM",   lab="DAM",         color="#D62728"),
    list(col="mean_SASP",  lab="SASP",        color="#FF7F0E")
)
for (p in PAIRS) stopifnot(p$col %in% colnames(plot_df))

# ─── shared stage bands ───
band_df <- do.call(rbind, lapply(sg_ordered, function(sg){
    d<-plot_df[plot_df$Study_Group==sg,]; if(nrow(d)==0) return(NULL)
    data.frame(xmin=min(d$x)-0.5, xmax=max(d$x)+0.5,
               fill=STUDY_GROUP_PALETTE[sg] %||% fallback_color) }))

# ─── panel builder ───
make_panel <- function(p, show_y=TRUE, show_x=FALSE) {
    var_y <- smooth_trace(z_score(plot_df[[p$col]]))
    d <- rbind(
        data.frame(x=plot_df$x, y=var_y, trace=p$lab,  color=p$color,   lw=0.9),
        data.frame(x=plot_df$x, y=cdr_y, trace="CDR",  color="#2CA02C", lw=0.9),
        data.frame(x=plot_df$x, y=snc_y, trace="%SnC", color="#6A0DAD", lw=1.5))
    # correlation of %SnC with this variable (annotation)
    rho <- suppressWarnings(cor(z_score(plot_df$pct_sen), z_score(plot_df[[p$col]]),
                                method="spearman", use="complete.obs"))
    ggplot() +
        geom_rect(data=band_df, aes(xmin=xmin,xmax=xmax,ymin=-Inf,ymax=Inf,fill=fill),
                  alpha=0.10, inherit.aes=FALSE) + scale_fill_identity() +
        geom_hline(yintercept=0, color="grey70", linetype="dashed", linewidth=0.3) +
        geom_line(data=d, aes(x=x,y=y,group=trace), color="white",
                  linewidth=d$lw+1.0, alpha=0.8, lineend="round") +
        geom_line(data=d, aes(x=x,y=y,color=color,group=trace,linewidth=lw), lineend="round") +
        annotate("text", x=Inf, y=Inf, hjust=1.08, vjust=1.4, size=2.6, color="#555555",
                 label=sprintf("rho[SnC,%s] == %+.2f", gsub("[^A-Za-z]","",p$lab), rho), parse=TRUE) +
        scale_color_identity() + scale_linewidth_identity() +
        scale_x_continuous(expand=c(0,0)) +
        labs(subtitle=p$lab,
             x=if(show_x) "Donors (group \u2192 Braak \u2192 CDR)" else NULL,
             y=if(show_y) "Score (z-score)" else NULL) +
        theme_classic(base_size=10) +
        theme(plot.subtitle=element_text(size=10, face="bold", color="#222222"),
              panel.border=element_rect(color="#333333", fill=NA, linewidth=0.5),
              axis.line=element_blank(),
              axis.text.x=element_blank(), axis.ticks.x=element_blank(),
              axis.text.y=element_text(size=8.5, color="#333333"),
              axis.title=element_text(size=9, color="#333333"),
              plot.margin=margin(6,8,6,8), legend.position="none")
}

# y-label left column only; x-label bottom row only
panels <- list(
    make_panel(PAIRS[[1]], show_y=TRUE,  show_x=FALSE),
    make_panel(PAIRS[[2]], show_y=FALSE, show_x=FALSE),
    make_panel(PAIRS[[3]], show_y=TRUE,  show_x=TRUE),
    make_panel(PAIRS[[4]], show_y=FALSE, show_x=TRUE)
)

# ─── two-row legend, CENTERED ───
leg_df <- data.frame(
    label = c("%SnC (senescent fraction)", "CDR (clinical severity)", "Homeostatic",
              "IFN (IRM)", "DAM", "SASP"),
    color = c("#6A0DAD","#2CA02C","#17BECF","#1F77B4","#D62728","#FF7F0E"),
    stringsAsFactors = FALSE)
leg_df$row <- rep(c(2,1), each=3)[seq_len(nrow(leg_df))]   # row 2 = top
leg_df$col <- rep(1:3, times=2)[seq_len(nrow(leg_df))]

slot_w   <- 3.2
n_cols   <- 3
total_w  <- n_cols * slot_w                 # full laid-out width
x_lim    <- c(0, total_w)                   # plot x-range
# center: the content already spans 0..total_w, so it's centered if x-limits match.
# but the LAST item's text extends past its slot — pad the limits symmetrically
text_pad <- 1.6                             # room for the longest label's text
leg_df$x_seg <- (leg_df$col-1)*slot_w + 0.1
leg_df$x_txt <- leg_df$x_seg + 0.35

legend_plot <- ggplot(leg_df) +
    geom_segment(aes(x=x_seg, xend=x_seg+0.28, y=row, yend=row, color=color),
                 linewidth=1.2, lineend="round") +
    geom_text(aes(x=x_txt, y=row, label=label, color=color),
              size=2.7, hjust=0, vjust=0.5) +
    scale_color_identity() +
    # symmetric x-limits center the content block; pad right for trailing text
    scale_x_continuous(limits=c(-text_pad, total_w + text_pad), expand=c(0,0)) +
    scale_y_continuous(limits=c(0.5, 2.5), expand=c(0,0)) +
    theme_void() + theme(plot.margin=margin(4,8,4,8))
             
# ─── compose + save ───
composed <- (wrap_plots(panels, ncol=2)) / legend_plot +
    plot_layout(heights=c(1, 0.12)) +
    plot_annotation(
        title = "Senescence relative to microglial activation programs across disease progression",
        subtitle = sprintf("%s \u00b7 %s \u00b7 %%SnC and CDR anchored in each panel \u00b7 n=%d donors",
                           CELL_TYPE, DATASET, nrow(plot_df)),
        theme = theme(plot.title=element_text(size=12, face="bold", color="#222222"),
                      plot.subtitle=element_text(size=8.5, color="#666666", margin=margin(b=4))))

save_figure(composed, slug=sprintf("%s_progression_paired", CELL_TYPE), width=10, height=6.5)
options(repr.plot.width=10, repr.plot.height=7); print(composed)

# ─── correlation report ───
cat("\nSpearman rho (%SnC vs each, donor-level):\n")
for (p in PAIRS) {
    rho <- suppressWarnings(cor(plot_df$pct_sen, plot_df[[p$col]], method="spearman", use="complete.obs"))
    cat(sprintf("  %%SnC vs %-12s rho = %+.3f\n", p$lab, rho))
}
rho_cdr <- cor(plot_df$pct_sen, plot_df$CDR, method="spearman", use="complete.obs")
cat(sprintf("  %%SnC vs %-12s rho = %+.3f\n", "CDR", rho_cdr))